# Complete Indian Traffic Sign Recognition (TSR) Training Pipeline
### Dual-Dataset (Kaushik + SDI) Integration & YOLOv8s Training on Google Colab (GPU)

This complete end-to-end pipeline:
1. **Verifies GPU acceleration** & installs dependencies.
2. **Downloads both authentic Indian TSR datasets** from Roboflow (~15,000 images).
3. **Sanitizes, merges, and aligns** all labels to the 15 canonical Indian traffic sign classes.
4. **Trains YOLOv8s** at 800px resolution (eliminating Pedestrian vs School confusion).
5. **Evaluates metrics** & plots confusion matrix.
6. **Auto-downloads `best.pt`** directly to your laptop.

In [ ]:
# Step 1: Check GPU Acceleration & Install Dependencies
!nvidia-smi
!pip install --upgrade ultralytics opencv-python matplotlib pyyaml roboflow

In [ ]:
# Step 2: Download Both Authentic Indian TSR Datasets (Kaushik + SDI)
from roboflow import Roboflow

rf = Roboflow(api_key="W5XwFcfOLfFEnyeTDfsG")

print('--- 1. Downloading Kaushik Dataset (Directional Signs, Curves, Horn Prohibited) ---')
p1 = rf.workspace("kaushik-gfstb").project("traffic-signs-pq69p")
p1.version(1).download("yolov8")

print('\n--- 2. Downloading SDI Dataset (Stop, No Entry, Speed Limits, Pedestrians) ---')
p2 = rf.workspace("sdi").project("indian-traffic-sign")
p2.version(2).download("yolov8")

print('\nBoth datasets downloaded successfully!')

In [ ]:
# Step 3: Strict Sanitizer & Merger (Zero Cross-Contamination)
import os, sys, glob, shutil, yaml
from pathlib import Path

CANONICAL_CLASSES = [
    'Stop', 'No Entry', 'Speed Limit 20', 'Speed Limit 40', 'Speed Limit 60',
    'Speed Limit 80', 'Give Way', 'School Ahead', 'Pedestrian Crossing',
    'Left Turn', 'Right Turn', 'U-Turn', 'Horn Prohibited', 'Parking', 'Dangerous Curve'
]

NUMERIC_MAP = {'14': 0, '17': 1, '0': 2, '20': 2, '40': 3, '3': 4, '60': 4, '5': 5, '80': 5, '13': 6, '28': 7, '27': 8, '34': 9, '64': 9, '72': 9, '39': 9, '37': 9, '33': 10, '63': 10, '73': 10, '38': 10, '36': 10, '53': 11, '60': 12, '65': 13, '19': 14, '20': 14, '21': 14, '43': 14, '44': 14, '45': 14, '46': 14, '47': 14, '48': 14, '57': 14}
JUNK_PREFIXES = ('*', '=', 'roboflow', 'the ', 'for ', 'to ', 'traffic-sign are', 'traffic sign -', 'visit ', 'annotate')

def map_label(name: str):
    s = str(name).strip().lower().replace('_', ' ')
    if any(s.startswith(pfx) for pfx in JUNK_PREFIXES) or len(s) > 35:
        return None
    if s.isdigit() and s in NUMERIC_MAP:
        return NUMERIC_MAP[s]
    if s == 'stop' or 'stop sign' in s:
        return 0
    if 'no entry' in s or 'all motor vehicle prohibited' in s:
        return 1
    if 'speed limit 20' in s or s == 'speed limit (20km/h)':
        return 2
    if 'speed limit 40' in s or s == 'speed limit (40km/h)':
        return 3
    if 'speed limit 60' in s or s == 'speed limit (60km/h)':
        return 4
    if 'speed limit 80' in s or s == 'speed limit (80km/h)':
        return 5
    if 'give way' in s or 'yield' in s:
        return 6
    if 'school ahead' in s or 'school' in s or 'children crossing' in s:
        return 7
    if ('pedestrian crossing' in s or 'zebra crossing' in s) and 'prohibited' not in s:
        return 8
    if any(k in s for k in ['turn left', 'left turn', 'compulsary keep left', 'compulsary ahead or turn left', 'compulsary turn left']) and 'prohibited' not in s:
        return 9
    if any(k in s for k in ['turn right', 'right turn', 'compulsary keep right', 'compulsary ahead or turn right', 'compulsary turn right']) and 'prohibited' not in s:
        return 10
    if ('u turn' in s or 'uturn' in s or 'gap in median' in s) and 'prohibited' not in s:
        return 11
    if 'horn prohibited' in s or 'no horn' in s:
        return 12
    if ('parking' in s or 'parking area' in s) and 'no parking' not in s:
        return 13
    if any(k in s for k in ['curve', 'bend', 'dip', 'barrier ahead']):
        return 14
    return None

candidate_dirs = [p for p in Path('/content').iterdir() if p.is_dir() and any(k in p.name.lower() for k in ['traffic', 'sign', 'sdi', 'indian']) and (p / 'data.yaml').exists() and 'clean' not in p.name.lower()]
print(f'[INFO] Merging {len(candidate_dirs)} datasets: {[d.name for d in candidate_dirs]}')

clean_dir = Path('/content/clean_indian_tsr')
for split in ['train', 'valid', 'test']:
    (clean_dir / split / 'images').mkdir(parents=True, exist_ok=True)
    (clean_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

class_counts = {i: 0 for i in range(15)}
total_boxes = 0

for sdir in candidate_dirs:
    print(f'  -> Sanitizing {sdir.name}...')
    with open(sdir / 'data.yaml', 'r', encoding='utf-8') as f:
        yd = yaml.safe_load(f)
        n = yd.get('names', {})
        names_dict = {i: str(v) for i, v in enumerate(n)} if isinstance(n, list) else {int(k): str(v) for k, v in n.items()}
    id_map = {oid: map_label(v) for oid, v in names_dict.items() if map_label(v) is not None}
    for split in ['train', 'valid', 'test']:
        src_split = split
        if not (sdir / src_split).exists() and split == 'valid':
            src_split = 'val'
        img_dir = (sdir / src_split / 'images') if (sdir / src_split / 'images').is_dir() else (sdir / src_split)
        lbl_dir = (sdir / src_split / 'labels') if (sdir / src_split / 'labels').is_dir() else (sdir / src_split)
        if not lbl_dir.exists():
            continue
        for tf in lbl_dir.glob('*.txt'):
            img_match = None
            for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG']:
                cand = img_dir / f'{tf.stem}{ext}'
                if cand.exists():
                    img_match = cand
                    break
            if not img_match:
                continue
            with open(tf, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    try:
                        old_c = int(float(parts[0]))
                        if old_c in id_map:
                            new_c = id_map[old_c]
                            new_lines.append(f'{new_c} {" ".join(parts[1:])}\n')
                            class_counts[new_c] += 1
                            total_boxes += 1
                    except ValueError:
                        pass
            if new_lines:
                pfx = sdir.name.replace('-', '_').lower()
                dest_img = clean_dir / split / 'images' / f'{pfx}_{img_match.name}'
                dest_lbl = clean_dir / split / 'labels' / f'{pfx}_{tf.name}'
                shutil.copy(img_match, dest_img)
                with open(dest_lbl, 'w', encoding='utf-8') as f:
                    f.writelines(new_lines)

clean_yaml_path = '/content/clean_indian_tsr/data.yaml'
clean_cfg = {
    'path': '/content/clean_indian_tsr',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 15,
    'names': {i: name for i, name in enumerate(CANONICAL_CLASSES)}
}
with open(clean_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(clean_cfg, f, default_flow_style=False, sort_keys=False)

print('\n' + '=' * 70)
print(f'[SUCCESS] CLEAN UNPOLLUTED 15-CLASS DATASET READY AT: {clean_yaml_path}')
print(f'Total Verified Bounding Boxes: {total_boxes}')
print('=' * 70)
for cid, cname in enumerate(CANONICAL_CLASSES):
    print(f'  Class {cid:2d} ({cname:<20}): {class_counts[cid]:6d} instances')


In [ ]:
# Step 4: Train High-Accuracy YOLOv8s on Tesla T4 GPU (50 Epochs, 800px)
from ultralytics import YOLO

# Load YOLOv8s (11.2M params) - eliminates silhouette confusion
model = YOLO('yolov8s.pt')

results = model.train(
    data='/content/clean_indian_tsr/data.yaml',
    epochs=50,
    imgsz=800,
    batch=16,
    optimizer='AdamW',
    lr0=0.001,
    device=0,
    project='tsr_colab_run',
    name='yolov8s_indian_tsr_clean',
    save=True,
    plots=True
)


In [ ]:
# Step 5: Evaluate Validation Metrics & Confusion Matrix
from IPython.display import Image, display
import glob

metrics = model.val()
print('\n--- 15-Class Validation Metrics ---')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

for plot_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    matches = glob.glob(f'/content/tsr_colab_run/yolov8s_indian_tsr_clean/{plot_name}')
    if matches:
        print(f'\n--- {plot_name} ---')
        display(Image(matches[0]))


In [ ]:
# Step 6: Automatically Locate & Download best.pt to Laptop
import glob, os, shutil
from google.colab import files

weight_candidates = glob.glob('/content/**/best.pt', recursive=True)
if weight_candidates:
    best_weights = max(weight_candidates, key=os.path.getmtime)
    print(f'[FOUND] Found newly trained weights at: {best_weights}')
    print('[DOWNLOAD] Triggering browser download to your laptop...')
    files.download(best_weights)
else:
    print('[ERROR] No best.pt file found on disk. Check Step 4 logs.')
